# **TaniMol: 02 - Fingerprint Generation**

This notebook loads the preprocessed dataset from `01_preprocessing` and generates molecular fingerprints for each compound. The fingerprints are binary bit vectors that encode structural features and will be used to compute pairwise Tanimoto similarity in the next step.

**Input:** `data/processed/cleaned_activities.csv`  
**Fingerprint types:** Morgan (ECFP4), MACCS keys, RDKit topological  
**Output:** fingerprint arrays ready for similarity computation

In [2]:
import pandas as pd
import numpy as np

from src.config import OUTPUT_PATH, MORGAN_FP_PATH, MACCS_FP_PATH, RDKIT_FP_PATH
from src.fingerprints import (
    add_fingerprints,
    save_fingerprints
)

### **1. Load Preprocessed Data**

Load the cleaned dataset produced by `01_preprocessing`. Each row is a unique (target, molecule) pair with a standardized SMILES and pIC50 value.

In [3]:
df = pd.read_csv(OUTPUT_PATH)

### **2. Generate Morgan Fingerprints (ECFP4)**

Morgan fingerprints with radius=2 (ECFP4) capture circular substructures around each atom up to 2 bonds away. This is the standard fingerprint for similarity-based analysis in drug discovery. Each molecule becomes a 2048-bit binary vector.

In [4]:
df_morgan = add_fingerprints(df, fp_type="morgan")
morgan_fps = df_morgan["morgan_fp"].tolist()

Generating morgan: 100%|██████████| 3324/3324 [00:00<00:00, 7734.75it/s]


### **3. Generate MACCS Keys**

MACCS keys use 166 predefined structural patterns (e.g. "contains aromatic ring", "has nitrogen"). Less granular than Morgan but useful for comparison — different fingerprint types can produce different similarity rankings.

In [5]:
df = add_fingerprints(df, fp_type="maccs")
maccs_fps = df["maccs_fp"].tolist()

Generating maccs: 100%|██████████| 3324/3324 [00:02<00:00, 1381.61it/s]


### **4. Generate RDKit Topological Fingerprints**

RDKit fingerprints encode topological paths (linear sequences of bonds) of various lengths. Path-based rather than circular — captures different structural information than Morgan.

In [6]:
df = add_fingerprints(df, fp_type="rdkit")
rdkit_fps = df["rdkit_fp"].tolist()

Generating rdkit: 100%|██████████| 3324/3324 [00:04<00:00, 742.96it/s]


### **5. Quick Sanity Check**

Verify that the fingerprints look reasonable: check bit density (fraction of \"on\" bits). Typical ECFP4 density is ~1-5% for drug-like molecules. For MACCS this number is around 33%.

In [7]:
for name, fps in [("Morgan", morgan_fps), ("MACCS", maccs_fps), ("RDKit", rdkit_fps)]:
    valid = [fp for fp in fps if fp is not None]
    avg_density = np.mean([fp.mean() for fp in valid]) * 100
    none_count = len(fps) - len(valid)
    print(f"{name}: Density: {avg_density:.2f}%, Failed: {none_count}/{len(fps)}")

Morgan: Density: 2.63%, Failed: 0/3324
MACCS: Density: 32.82%, Failed: 0/3324
RDKit: Density: 49.67%, Failed: 0/3324


### **6. Save Fingerprint Arrays**

Save all three fingerprint types to `data/processed/fingerprints/` as `.npy` files. These can be loaded directly in downstream notebooks (similarity, clustering) without regenerating.

In [8]:
save_fingerprints(morgan_fps, MORGAN_FP_PATH)
save_fingerprints(maccs_fps, MACCS_FP_PATH)
save_fingerprints(rdkit_fps, RDKIT_FP_PATH)

Saved 3324 fingerprints to /home/stanuch/Dev/TaniMol/data/processed/fingerprints/morgan_fps.npy
Saved 3324 fingerprints to /home/stanuch/Dev/TaniMol/data/processed/fingerprints/maccs_fps.npy
Saved 3324 fingerprints to /home/stanuch/Dev/TaniMol/data/processed/fingerprints/rdkit_fps.npy
